# Lab 01: Find the Best Classification Model with AutoML + MLflow

> **Source:** Microsoft Learning - mslearn-mlops
> **Original files:** [github.com/MicrosoftLearning/mslearn-mlops](https://github.com/MicrosoftLearning/mslearn-mlops)
> **License:** MIT
> **Adapted for:** AI-300 Lab Guide

## Learning Objectives

By the end of this lab, you will be able to:

1. **Submit an AutoML job** using the Azure ML Python SDK v2
2. **Track experiments with MLflow** using both autologging and custom logging
3. **Compare models** in Azure ML Studio and review metrics, parameters, and artifacts

**Estimated time:** ~30 minutes
**Estimated cost:** ~$1-2 (compute cluster usage)

## Architecture

![Architecture](architecture.png)

**Part 1** uses AutoML to automatically try multiple classification algorithms on a compute cluster.
**Part 2** uses MLflow to manually train and track models locally on the compute instance.

## Prerequisites

Before running this lab, ensure you have completed the initial setup:

```bash
bash scripts/setup-mlops.sh
```

This script provisions the Azure ML workspace, compute cluster, and data assets needed for this lab.

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
ws = ml_client.workspaces.get(ml_client.workspace_name)
print(f"Workspace: {ws.name} ({ws.location})")
compute = ml_client.compute.get("cc-ai300")
print(f"Compute cluster: {compute.name} ({compute.size})")
print("\nAll prerequisites met.")

## Part 1: AutoML Classification

**Automated Machine Learning (AutoML)** lets you try multiple algorithms and preprocessing transformations with your data -- automatically. It trains and tunes many models, then recommends the best one based on your chosen metric.

For classification, AutoML will try algorithms like LightGBM, XGBoost, Random Forest, Decision Trees, SGD, and more.

> **Exam Tip:** The `primary_metric` parameter determines how models are ranked. Common options for classification:
> - `accuracy` -- proportion of correct predictions (good for balanced datasets)
> - `AUC_weighted` -- area under the ROC curve, weighted by class (good for imbalanced datasets)
> - `norm_macro_recall` -- recall averaged across classes, normalized (good for imbalanced datasets)

### Load the training data

AutoML requires data in **MLTable** format, which includes a schema definition file (`MLTable`) alongside the data file(s). This tells AutoML the column types and how to read the data.

> **Exam Tip:** Know the difference between data asset types:
> - **`MLTable`** -- includes schema definition, used by AutoML
> - **`uri_file`** -- points to a single file (CSV, Parquet, etc.)
> - **`uri_folder`** -- points to a folder of files

In [ ]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

# Data source: Microsoft Learning - mslearn-mlops
# Original: https://github.com/MicrosoftLearning/mslearn-mlops/tree/main/data/diabetes-data
my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:diabetes-training:1")
print(f"Data asset: {my_training_data_input.path}")

### Configure and submit the AutoML job

Key configuration parameters:

| Parameter | Description |
|-----------|-------------|
| `compute` | The compute target to run trials on (must be a **cluster**, not an instance) |
| `target_column_name` | The label column to predict |
| `primary_metric` | The metric to optimize (`accuracy`, `AUC_weighted`, `norm_macro_recall`) |
| `n_cross_validations` | Number of cross-validation folds |
| `set_limits()` | Controls cost: `timeout_minutes`, `max_trials`, `enable_early_termination` |
| `blocked_training_algorithms` | Algorithms to exclude from trials |

> **Exam Tip:** `enable_early_termination=True` stops trials that are not improving, saving cost and time. This is a best practice for exam scenarios asking about cost optimization.

In [ ]:
from azure.ai.ml import automl

classification_job = automl.classification(
    compute="cc-ai300",
    experiment_name="auto-ml-class-dev",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="accuracy",
    n_cross_validations=5,
    enable_model_explainability=True,
)
classification_job.set_limits(
    timeout_minutes=60, trial_timeout_minutes=20,
    max_trials=5, enable_early_termination=True,
)
classification_job.set_training(
    blocked_training_algorithms=["LogisticRegression"],
    enable_onnx_compatible_models=True,
)
returned_job = ml_client.jobs.create_or_update(classification_job)
print(f"AutoML job submitted: {returned_job.name}")
print(f"Monitor at: {returned_job.studio_url}")

### Review AutoML results in Azure ML Studio

Once the job completes (typically 15-30 minutes), review the results:

1. Click the **Studio URL** printed above, or navigate to [ml.azure.com](https://ml.azure.com)
2. Go to **Jobs** > **auto-ml-class-dev** > click the latest run
3. Review the **Models** tab to see all tried algorithms ranked by accuracy
4. Click the **best model** to see:
   - **Metrics** -- accuracy, AUC, precision, recall, F1
   - **Explanations** -- feature importance (because we set `enable_model_explainability=True`)
   - **Images** -- confusion matrix, ROC curve, calibration plot

> **Exam Tip:** AutoML jobs can only run on **compute clusters**, not compute instances. Compute instances are used for development (notebooks, scripts), while compute clusters scale out for training jobs.

## Part 2: Track Models with MLflow

**MLflow** is the industry-standard open-source platform for managing the ML lifecycle. Azure ML has a **built-in MLflow tracking server** -- no extra setup required.

MLflow provides three logging approaches:
1. **Autologging** -- automatically captures parameters, metrics, and model artifacts
2. **Custom logging** -- manual `log_param()`, `log_metric()`, `log_artifact()` calls
3. **Combined** -- autologging + custom metrics for maximum visibility

### Prepare the data

For MLflow tracking, we'll work with the CSV data directly (no MLTable needed). We load, split, and prepare features/labels for scikit-learn models.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Data source: Microsoft Learning - mslearn-mlops
df = pd.read_csv("data/diabetes.csv")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

X = df[["Pregnancies", "PlasmaGlucose", "DiastolicBloodPressure", "TricepsThickness",
         "SerumInsulin", "BMI", "DiabetesPedigree", "Age"]].values
y = df["Diabetic"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")

### MLflow Autologging

`mlflow.sklearn.autolog()` automatically captures:
- **Parameters** -- all constructor arguments (C, solver, max_iter, etc.)
- **Metrics** -- training score
- **Model artifact** -- the serialized model (pickle)
- **Model signature** -- input/output schema

> **Exam Tip:** Autologging works with many frameworks: `mlflow.sklearn`, `mlflow.tensorflow`, `mlflow.pytorch`, `mlflow.xgboost`, `mlflow.lightgbm`, `mlflow.spark`. The exam may ask which frameworks support autologging.

In [ ]:
import mlflow
from sklearn.linear_model import LogisticRegression

mlflow.set_experiment("mlflow-experiment-diabetes")

with mlflow.start_run():
    mlflow.sklearn.autolog()
    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)

print("Run 1 complete: LogisticRegression with autologging")

### MLflow Custom Logging

For more control, disable autologging and log exactly what you need:

- `mlflow.log_param("name", value)` -- log a single hyperparameter
- `mlflow.log_metric("name", value)` -- log a single metric
- `mlflow.log_artifact("path/to/file")` -- log a file (plot, data, config, etc.)

This gives you fine-grained control over what appears in your experiment tracking.

In [ ]:
import numpy as np

mlflow.sklearn.autolog(disable=True)

with mlflow.start_run():
    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    mlflow.log_param("regularization_rate", 0.1)
    mlflow.log_metric("Accuracy", acc)
print(f"Run 2 complete: Accuracy = {acc:.4f}")

with mlflow.start_run():
    model = LogisticRegression(C=1/0.01, solver="liblinear").fit(X_train, y_train)
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    mlflow.log_param("regularization_rate", 0.01)
    mlflow.log_metric("Accuracy", acc)
print(f"Run 3 complete: Accuracy = {acc:.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier

with mlflow.start_run():
    model = DecisionTreeClassifier().fit(X_train, y_train)
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)
print(f"Run 4 complete: Accuracy = {acc:.4f}")

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

with mlflow.start_run():
    model = DecisionTreeClassifier().fit(X_train, y_train)
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    y_scores = model.predict_proba(X_test)
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:, 1])

    fig = plt.figure(figsize=(6, 4))
    plt.plot([0, 1], [0, 1], "k--")
    plt.plot(fpr, tpr)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.savefig("ROC-Curve.png")
    plt.show()

    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)
    mlflow.log_artifact("ROC-Curve.png")

print(f"Run 5 complete: Accuracy = {acc:.4f} (with ROC artifact)")

## Key Takeaways

1. **AutoML automates model selection** -- it tries multiple algorithms and hyperparameter combinations, then ranks them by your chosen `primary_metric`
2. **MLflow is the standard for experiment tracking** -- Azure ML has a built-in MLflow tracking server; use `autolog()` for convenience or `log_param()`/`log_metric()`/`log_artifact()` for fine-grained control
3. **Data asset types matter** -- AutoML requires `MLTable` format; command jobs can use `uri_file` or `uri_folder`
4. **Compute clusters scale for training** -- AutoML and command jobs run on clusters (`cc-ai300`), while notebooks run on compute instances
5. **Cost controls are essential** -- use `set_limits()` with `timeout_minutes`, `max_trials`, and `enable_early_termination` to manage spend